# Multi-Robot Path Finding: Cooperative A* vs POPS vs PBS

This notebook compares the three algorithms across **robot counts** and **obstacle probabilities** on generated grids.

- **Cooperative A\***: fixed input priority order.
- **POPS**: exhaustively tests every priority permutation for each instance (up to 7 robots, so at most 5,040 orders per instance), then selects the best complete schedule by makespan and total path length.
- **PBS**: explores partial priority constraints generated by conflicts.

The notebook saves a checkpoint CSV after every instance. It prints progress while POPS evaluates permutations.

**Experiment defaults:** 10×10 grids, robot counts 2–7, obstacle probabilities 0.10/0.20/0.30, one generated instance per combination. Increase `REPEATS` if you have time. Exhaustive POPS for 7 robots means 5,040 planner runs *per generated instance*, so the total workload can still be substantial.


In [ ]:
import os
import time
import random
import itertools
import math
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from cooperative_astar import prioritized_cooperative_astar
from pbs_mapf import priority_based_search

# ---------------- EXPERIMENT SETTINGS ----------------
GRID_ROWS = 10
GRID_COLS = 10
ROBOT_COUNTS = [2, 3, 4, 5, 6, 7]
OBSTACLE_PROBS = [0.10, 0.20, 0.30]
REPEATS = 1
BASE_SEED = 42

# PBS's high-level search budget. Increase for harder instances if needed.
PBS_HIGH_LEVEL_LIMIT = 5000

# None uses the search horizon chosen by the algorithm.
MAX_TIME = None

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(RESULTS_DIR, "mapf_sweep_comparison.csv")

print("Configuration loaded.", flush=True)
print(f"Instances planned: {len(ROBOT_COUNTS) * len(OBSTACLE_PROBS) * REPEATS}")
print(f"POPS tests all n! priority orders: up to {len(ROBOT_COUNTS)} robots means 5,040 orders per 7-robot instance.", flush=True)


## 1. Grid generation

Each generated grid has unique start cells and unique goal cells. We reject a grid unless each robot has an individual obstacle-free route from its start to its goal. This checks individual reachability, not whether all robots can necessarily be routed together without collisions.


In [ ]:
def individually_reachable(grid, start, goal):
    rows, cols = grid.shape
    q = deque([start])
    visited = {start}
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while q:
        cell = q.popleft()
        if cell == goal:
            return True
        for dr, dc in directions:
            nxt = (cell[0] + dr, cell[1] + dc)
            if (0 <= nxt[0] < rows and 0 <= nxt[1] < cols
                    and grid[nxt] == 0 and nxt not in visited):
                visited.add(nxt)
                q.append(nxt)
    return False


def generate_instance(rows, cols, obstacle_prob, n_robots, seed, max_attempts=1000):
    rng = np.random.default_rng(seed)
    py_rng = random.Random(seed)
    total = rows * cols
    if 2 * n_robots > total:
        raise ValueError("Not enough cells for unique starts and goals.")
    for attempt in range(max_attempts):
        grid = rng.choice([0, 1], size=(rows, cols),
                          p=[1 - obstacle_prob, obstacle_prob]).astype(int)
        positions = py_rng.sample(range(total), 2 * n_robots)
        starts = [(p // cols, p % cols) for p in positions[:n_robots]]
        goals = [(p // cols, p % cols) for p in positions[n_robots:]]
        for cell in starts + goals:
            grid[cell] = 0
        if all(individually_reachable(grid, starts[i], goals[i])
               for i in range(n_robots)):
            return grid, starts, goals
    raise RuntimeError(
        f"Could not generate a valid {rows}x{cols} grid at obstacle probability "
        f"{obstacle_prob}. Try a lower probability or larger grid."
    )


def show_instance(grid, starts, goals, title="Generated MAPF instance"):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(grid, cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    ax.grid(True, linewidth=0.5)
    for i, (s, g) in enumerate(zip(starts, goals), start=1):
        ax.scatter(s[1], s[0], marker="o", s=90, label=f"R{i} start")
        ax.scatter(g[1], g[0], marker="*", s=150, label=f"R{i} goal")
        ax.text(s[1], s[0], str(i), ha="center", va="center", fontsize=8)
    ax.set_title(title)
    ax.invert_yaxis()
    ax.legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


## 2. Metric helpers

Makespan and total path length are only reported when **all robots succeed**. Success rate is reported separately. Runtime includes the algorithm's full search time; POPS runtime includes all tested priority orders.


In [ ]:
def summarize_results(algorithm, results, n_robots, runtime_s, instance_id,
                      obstacle_prob, extra=""):
    if results is None:
        success_count = 0
        robots = []
    else:
        robots = [results.get(f"Robot {i+1}", {}) for i in range(n_robots)]
        success_count = sum(bool(r.get("success")) for r in robots)

    complete = success_count == n_robots
    arrivals = [r.get("arrival_time") for r in robots
                if r.get("success") and r.get("arrival_time") is not None]
    lengths = [r.get("path_length") for r in robots
               if r.get("success") and r.get("path_length") is not None]
    nodes = sum(r.get("nodes_expanded", 0) for r in robots)

    return {
        "instance_id": instance_id,
        "robots": n_robots,
        "obstacle_probability": obstacle_prob,
        "algorithm": algorithm,
        "solved_robots": success_count,
        "success_rate_percent": round(100 * success_count / n_robots, 2),
        "all_succeeded": complete,
        "makespan": max(arrivals) if complete and arrivals else np.nan,
        "total_path_length": sum(lengths) if complete else np.nan,
        "nodes_expanded": nodes,
        "runtime_seconds": round(runtime_s, 6),
        "details": extra
    }


def save_checkpoint(rows):
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)


## 3. Run the full sweep

The notebook tests every permutation for POPS. For 7 robots this is 5,040 orders per instance. A progress line is printed every 100 orders. Results are checkpointed after each algorithm and each instance.

If you need a faster test run first, temporarily set `ROBOT_COUNTS = [2, 3]`, `OBSTACLE_PROBS = [0.10]`, and run all cells. Then restore the full configuration.


In [ ]:
rows = []
total_instances = len(ROBOT_COUNTS) * len(OBSTACLE_PROBS) * REPEATS
instance_number = 0
sweep_start = time.perf_counter()

for obstacle_prob in OBSTACLE_PROBS:
    for n_robots in ROBOT_COUNTS:
        for repeat in range(REPEATS):
            instance_number += 1
            seed = BASE_SEED + int(obstacle_prob * 1000) * 100 + n_robots * 10 + repeat
            instance_id = f"p{obstacle_prob:.2f}_r{n_robots}_rep{repeat+1}"
            print(
                f"\n=== Instance {instance_number}/{total_instances}: "
                f"{instance_id} (seed={seed}) ===",
                flush=True
            )
            grid, starts, goals = generate_instance(
                GRID_ROWS, GRID_COLS, obstacle_prob, n_robots, seed
            )
            print(f"Grid {grid.shape}; obstacles={int(grid.sum())}; robots={n_robots}", flush=True)

            # A. Fixed-priority Cooperative A*
            print("[1/3] Cooperative A* running...", flush=True)
            t0 = time.perf_counter()
            fixed_results = prioritized_cooperative_astar(
                grid, starts, goals, max_time=MAX_TIME
            )
            elapsed = time.perf_counter() - t0
            rows.append(summarize_results(
                "Cooperative A*", fixed_results, n_robots, elapsed,
                instance_id, obstacle_prob, "Fixed input order"
            ))
            save_checkpoint(rows)
            print(f"      done in {elapsed:.2f}s", flush=True)

            # B. POPS: exhaustive permutation portfolio
            print(f"[2/3] POPS testing all {n_robots}! = {math.factorial(n_robots):,} orders...", flush=True)
            t0 = time.perf_counter()
            best_results = None
            best_order = None
            best_key = None
            complete_orders = 0
            order_count = 0

            for order in itertools.permutations(range(n_robots)):
                order_count += 1
                trial_results = prioritized_cooperative_astar(
                    grid, starts, goals, max_time=MAX_TIME,
                    priority_order=list(order)
                )
                success_count = sum(
                    bool(trial_results[f"Robot {i+1}"].get("success"))
                    for i in range(n_robots)
                )
                all_success = success_count == n_robots
                arrivals = [
                    trial_results[f"Robot {i+1}"].get("arrival_time")
                    for i in range(n_robots)
                    if trial_results[f"Robot {i+1}"].get("success")
                ]
                lengths = [
                    trial_results[f"Robot {i+1}"].get("path_length")
                    for i in range(n_robots)
                    if trial_results[f"Robot {i+1}"].get("success")
                ]
                makespan = max(arrivals) if all_success and arrivals else float("inf")
                total_length = sum(lengths) if all_success else float("inf")
                key = (0 if all_success else 1,
                       0 if all_success else -success_count,
                       makespan, total_length)
                if all_success:
                    complete_orders += 1
                if best_key is None or key < best_key:
                    best_key, best_results, best_order = key, trial_results, order

                if order_count % 100 == 0 or order_count == math.factorial(n_robots):
                    print(f"      orders {order_count}/{math.factorial(n_robots)}", flush=True)

            elapsed = time.perf_counter() - t0
            rows.append(summarize_results(
                "POPS", best_results, n_robots, elapsed, instance_id,
                obstacle_prob,
                f"All {order_count} orders tested; {complete_orders} complete schedules; "
                f"best order={[i+1 for i in best_order] if best_order is not None else None}"
            ))
            save_checkpoint(rows)
            print(f"      done in {elapsed:.2f}s; complete schedules={complete_orders}", flush=True)

            # C. PBS
            print(f"[3/3] PBS running (high-level limit={PBS_HIGH_LEVEL_LIMIT})...", flush=True)
            t0 = time.perf_counter()
            pbs_output = priority_based_search(
                grid, starts, goals, max_time=MAX_TIME,
                max_high_level_nodes=PBS_HIGH_LEVEL_LIMIT
            )
            elapsed = time.perf_counter() - t0
            rows.append(summarize_results(
                "PBS", pbs_output.get("results"), n_robots, elapsed,
                instance_id, obstacle_prob,
                f"High-level nodes={pbs_output.get('high_level_nodes')}; "
                f"status={pbs_output.get('reason')}"
            ))
            save_checkpoint(rows)
            print(f"      PBS done in {elapsed:.2f}s; {pbs_output.get('reason')}", flush=True)

print(f"\nSweep complete in {(time.perf_counter()-sweep_start)/60:.2f} minutes.")
results_df = pd.DataFrame(rows)
display(results_df)
print(f"Checkpoint/results CSV: {OUTPUT_CSV}")


## 4. Summarize and plot results

The charts aggregate the generated instances for each robot count and obstacle probability. With `REPEATS = 1`, each point represents one instance, so treat trends as preliminary; use more repeats for stronger conclusions.


In [ ]:
results_df = pd.DataFrame(rows)
if results_df.empty:
    print("No results yet. Run the sweep cell first.")
else:
    # Average metrics over repeats, preserving missing makespans when no complete schedule was found.
    summary = (results_df.groupby(
        ["robots", "obstacle_probability", "algorithm"], as_index=False
    ).agg(
        success_rate_percent=("success_rate_percent", "mean"),
        makespan=("makespan", "mean"),
        total_path_length=("total_path_length", "mean"),
        runtime_seconds=("runtime_seconds", "mean")
    ))
    display(summary)

    for metric, ylabel in [
        ("makespan", "Makespan (timesteps; complete schedules only)"),
        ("runtime_seconds", "Runtime (seconds)"),
        ("success_rate_percent", "Robots solved (%)")
    ]:
        fig, ax = plt.subplots(figsize=(9, 5))
        for prob in OBSTACLE_PROBS:
            subset = summary[summary["obstacle_probability"] == prob]
            for algorithm in ["Cooperative A*", "POPS", "PBS"]:
                line = subset[subset["algorithm"] == algorithm].sort_values("robots")
                if not line.empty:
                    ax.plot(line["robots"], line[metric], marker="o",
                            label=f"{algorithm}, p={prob:.2f}")
        ax.set_xlabel("Number of robots")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel} vs robot count and obstacle probability")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        plt.show()

    summary.to_csv(os.path.join(RESULTS_DIR, "mapf_sweep_summary.csv"), index=False)
    print("Saved summary to results/mapf_sweep_summary.csv")


## 5. Notes for interpreting results

- POPS exhaustively searches **priority orderings**, not every possible joint path schedule. It therefore does not guarantee globally optimal MAPF makespan.
- PBS can terminate at the high-level node limit. A failure under this limit is not proof that no solution exists.
- The generator guarantees individual start-to-goal reachability, not joint collision-free solvability.
- Makespan and total path length are `NaN` unless all robots succeed; this avoids misleading comparisons from incomplete schedules.
- Since only one instance is used by default per combination, do not make strong statistical claims. Increase `REPEATS` (for example to 3–5) if runtime permits.
